In [ ]:
import os
import geopandas as gpd

base = "/Users/nico/Documents/PhD research/energy poverty/GDL Shapefiles V4"
shp_path = os.path.join(base, "GDL Shapefiles V4.shp")

gdl_shp = gpd.read_file(shp_path)

print("CRS:", gdl_shp.crs)
print("Columns:")
print(gdl_shp.columns.tolist())

print("\nHead:")
print(gdl_shp.head())

for c in gdl_shp.columns:
    if "GDL" in c.upper() or "CODE" in c.upper() or "ID" in c.upper() or "NAME" in c.upper():
        print("\nCOLUMN:", c)
        print(gdl_shp[c].head(20).tolist())

In [ ]:
import os
import pandas as pd
import geopandas as gpd

csv_path = "/Users/nico/Documents/PhD research/energy poverty/global_combined_HDD.csv"
shp_path = "/Users/nico/Documents/PhD research/energy poverty/GDL Shapefiles V4/GDL Shapefiles V4.shp"

df = pd.read_csv(csv_path, usecols=["GDLcode"])
gdl_csv = set(df["GDLcode"].dropna().unique())

gdf = gpd.read_file(shp_path)
gdl_shp = set(gdf["GDLcode"].dropna().unique())

print("CSV unique GDLcode:", len(gdl_csv))
print("SHP unique GDLcode:", len(gdl_shp))
print("Matched:", len(gdl_csv & gdl_shp))
print("Only in CSV:", len(gdl_csv - gdl_shp))
print("Only in SHP:", len(gdl_shp - gdl_csv))

print("\nExamples only in CSV:")
print(sorted(list(gdl_csv - gdl_shp))[:20])

print("\nExamples only in SHP:")
print(sorted(list(gdl_shp - gdl_csv))[:20])

In [ ]:
import geopandas as gpd
import rasterio

shp_path = "/Users/nico/Documents/PhD research/energy poverty/GDL Shapefiles V4/GDL Shapefiles V4.shp"
tif_path = "/Users/nico/Documents/PhD research/energy poverty/EC2015.tif"

gdf = gpd.read_file(shp_path)
print("Original shapefile CRS:", gdf.crs)
print("Original shapefile bounds:", gdf.total_bounds)

gdf_proj = gdf.to_crs("ESRI:54009")
print("Projected shapefile bounds:", gdf_proj.total_bounds)

with rasterio.open(tif_path) as src:
    print("Raster CRS:", src.crs)
    print("Raster bounds:", src.bounds)
    print("Raster nodata:", src.nodata)

In [ ]:
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
from tqdm.auto import tqdm

csv_path = "/Users/nico/Documents/PhD research/energy poverty/global_combined_HDD.csv"
shp_path = "/Users/nico/Documents/PhD research/energy poverty/GDL Shapefiles V4/GDL Shapefiles V4.shp"
tif_path = "/Users/nico/Documents/PhD research/energy poverty/EC2015.tif"
out_path = "/Users/nico/Documents/PhD research/energy poverty/global_combined_HDD_with_EC2015.csv"

# 1. read data
df = pd.read_csv(csv_path)
gdf = gpd.read_file(shp_path)

# 2. keep only units used in CSV
gdl_csv = set(df["GDLcode"].dropna().unique())
gdf = gdf[gdf["GDLcode"].isin(gdl_csv)].copy()

print("Number of GDL units to extract:", len(gdf))

# 3. project to raster CRS
gdf = gdf.to_crs("ESRI:54009").reset_index(drop=True)

# 4. extract raster values
results = []

with rasterio.open(tif_path) as src:
    for i in tqdm(range(len(gdf)), desc="Extracting EC2015"):
        row = gdf.iloc[i]
        try:
            out_image, _ = mask(
                src,
                [row.geometry],
                crop=True,
                filled=False
            )
            arr = out_image[0]
            valid = arr.compressed()

            if valid.size == 0:
                results.append({
                    "GDLcode": row["GDLcode"],
                    "EC2015_sum": np.nan,
                    "EC2015_mean": np.nan,
                    "EC2015_pixel_count": 0
                })
            else:
                results.append({
                    "GDLcode": row["GDLcode"],
                    "EC2015_sum": float(valid.sum()),
                    "EC2015_mean": float(valid.mean()),
                    "EC2015_pixel_count": int(valid.size)
                })
        except Exception:
            results.append({
                "GDLcode": row["GDLcode"],
                "EC2015_sum": np.nan,
                "EC2015_mean": np.nan,
                "EC2015_pixel_count": 0
            })

stats_df = pd.DataFrame(results)

# 5. merge stats back to attributes
gdf_out = gdf.merge(stats_df, on="GDLcode", how="left")

# 6. merge back to CSV
merge_cols = [
    "GDLcode", "iso_code", "country", "region",
    "EC2015_sum", "EC2015_mean", "EC2015_pixel_count"
]
df_out = df.merge(gdf_out[merge_cols], on="GDLcode", how="left")

# 7. per capita electricity
df_out["EC2015_pc"] = df_out["EC2015_sum"] / df_out["population"]

# 8. save
df_out.to_csv(out_path, index=False)

print("\nSaved to:", out_path)

# 9. check missing
missing = df_out[["GDLcode", "country", "EC2015_sum"]].drop_duplicates()
missing = missing[missing["EC2015_sum"].isna()]

print("\nMissing GDL units:", len(missing))
if len(missing) > 0:
    print(missing["country"].value_counts())
else:
    print("No missing GDL units.")

In [ ]:
import pandas as pd
import numpy as np

path = "/Users/nico/Documents/PhD research/energy poverty/global_combined_HDD_with_EC2015.csv"
df = pd.read_csv(path)

# 只取 2015，一条 GDLcode 保留一行
df2015 = df[df["time"].astype(str).str.startswith("2015_")].copy()
df2015 = df2015.sort_values(["GDLcode", "time"]).groupby("GDLcode", as_index=False).first()

print("rows:", len(df2015))
print("\nColumns:")
print(df2015.columns.tolist())

print("\nElectricity summary:")
print(df2015[["EC2015_sum", "EC2015_mean", "EC2015_pixel_count", "EC2015_pc"]].describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]))

print("\nPotential group variables summary:")
cols = ["population", "age", "education", "gender", "Hhsize", "income", "rural", "wealth"]
print(df2015[cols].describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]))

print("\nMissing counts:")
print(df2015[cols + ["EC2015_sum", "EC2015_pc"]].isna().sum())

In [ ]:
for c in ["income", "education", "gender", "rural", "wealth"]:
    print("\n====", c, "====")
    print("nunique:", df2015[c].nunique(dropna=True))
    print("small sample uniques:", sorted(df2015[c].dropna().unique())[:20])
    print("min/max:", df2015[c].min(), df2015[c].max())